# **Интерактивная визуализация данных с Plotly / Interactive Data Visualization with Plotly**

# **Plotly** is a Python library for creating interactive visualizations. Unlike static Matplotlib charts, Plotly supports zooming, panning, hover tooltips, and interactive control over displayed data series.
# **Plotly** — библиотека Python для создания интерактивных графиков. В отличие от статических графиков Matplotlib, Plotly позволяет масштабировать изображения, перемещаться по графику, просматривать значения при наведении курсора и включать или отключать отдельные категории данных.


In [1]:
%pip install -U plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 22.4 MB/s eta 0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1


In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [3]:
df_clean = pd.read_csv("University Students 2026 New.csv")

# **Круговая диаграмма / Pie Chart**

In [4]:
program_counts = (
    df_clean["program"]
    .value_counts()
    .reset_index()
)
program_counts.columns = ["program", "students"]
fig = px.pie(
    program_counts,
    names="program",
    values="students",
    title="Distribution of Students by Program",
    hole=0.3
)

fig.update_traces(
    textinfo="percent+label"
)

fig.show()

# hole=0.3 создаёт отверстие в центре диаграммы, превращая её в кольцевую (donut chart). textinfo="percent+label" выводит название категории и её процент.
# hole=0.3 creates a donut chart. textinfo="percent+label" displays category names and percentages.

# **Корреляционная тепловая карта / Correlation Heatmap**

# Тепловая карта корреляций показывает силу и направление линейной связи между числовыми признаками. Значения коэффициента Пирсона находятся в диапазоне от −1 до +1.
# A correlation heatmap visualizes the strength and direction of linear relationships between numerical features. Pearson correlation coefficients range from −1 to +1.

In [5]:
numeric_columns = [
    "admission_score",
    "math_score",
    "programming_score",
    "english_score",
    "attendance",
    "assignments",
    "final_score"
]

corr_matrix = df_clean[numeric_columns].corr()

fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    aspect="auto",
    title="Correlation Matrix of Student Performance"
)

fig.update_layout(
    width=950,
    height=650
)

fig.show()

# text_auto=".2f" отображает коэффициенты с двумя знаками после запятой, а color_continuous_scale задаёт цветовую шкалу.
# text_auto=".2f" displays coefficients with two decimal places, while color_continuous_scale defines the color scale.

# **Диаграмма рассеивания / Scatter Plot**

In [6]:
fig = px.scatter(
    df_clean,
    x="attendance",
    y="final_score",
    color="program",
    hover_data=["student_id", "year"],
    title="Attendance vs Final Score",
    labels={
        "attendance": "Attendance (%)",
        "final_score": "Final Score",
        "program": "Academic Program"
    }
)

fig.update_layout(
    width=950,
    height=550
)

fig.show()

# hover_data добавляет информацию, отображаемую при наведении курсора на точку.
# hover_data specifies additional information displayed when hovering over a point.

# **Трёхмерная диаграмма рассеивания / 3D Scatter Plot**

In [7]:
fig = px.scatter_3d(
    df_clean,
    x="math_score",
    y="programming_score",
    z="final_score",
    color="program",
    hover_name="student_id",
    title="3D Student Performance Analysis",
    labels={
        "math_score": "Math Score",
        "programming_score": "Programming Score",
        "final_score": "Final Score"
    }
)

fig.update_layout(
    width=950,
    height=650
)

fig.show()

# **Временные ряды / Time Series**

In [8]:
df_clean["admission_date"] = pd.to_datetime(
    df_clean["admission_date"]
)
df_clean["admission_year"] = (
    df_clean["admission_date"].dt.year
)

In [9]:
yearly_scores = (
    df_clean
    .groupby(["admission_year", "program"], as_index=False)
    .agg(avg_score=("final_score", "mean"))
)

In [10]:
fig = px.line(
    yearly_scores,
    x="admission_year",
    y="avg_score",
    color="program",
    markers=True,
    title="Average Final Score by Admission Year",
    labels={
        "admission_year": "Admission Year",
        "avg_score": "Average Final Score",
        "program": "Academic Program"
    }
)

fig.update_xaxes(dtick=1)

fig.show()

# **Гистограмма / Histogram**

In [11]:
fig = px.histogram(
    df_clean,
    x="final_score",
    nbins=15,
    color="program",
    barmode="overlay",
    opacity=0.65,
    title="Distribution of Final Scores",
    labels={
        "final_score": "Final Score",
        "count": "Number of Students"
    }
)

fig.update_layout(
    xaxis_title="Final Score",
    yaxis_title="Number of Students",
    width=950,
    height=550
)

fig.show()

# nbins=15 задаёт желаемое количество интервалов, barmode="overlay" накладывает распределения друг на друга, а opacity=0.65 делает столбцы полупрозрачными.
# nbins=15 specifies the desired number of bins, barmode="overlay" overlays the distributions, and opacity=0.65 makes the bars partially transparent.

# **Ящик с усами / Boxplot**

In [13]:
fig = px.box(
    df_clean,
    x="program",
    y="final_score",
    color="program",
    #points="all",
    title="Final Score Distribution by Program",
    labels={
        "program": "Academic Program",
        "final_score": "Final Score"
    }
)

fig.update_layout(
    width=950,
    height=550,
    showlegend=False
)

fig.show()

# **Анимация графиков / Animations in Python**

In [14]:
animation_data = (
    df_clean
    .groupby(["year", "program"], as_index=False)
    .agg(students=("student_id", "count"))
)

programs = sorted(df_clean["program"].unique())
years = sorted(df_clean["year"].unique())

complete_index = pd.MultiIndex.from_product(
    [years, programs],
    names=["year", "program"]
)

animation_data = (
    animation_data
    .set_index(["year", "program"])
    .reindex(complete_index, fill_value=0)
    .reset_index()
)

In [16]:
fig = px.bar(
    animation_data,
    x="program",
    y="students",
    color="program",
    animation_frame="year",
    range_y=[0, animation_data["students"].max() * 1.15],
    title="Student Distribution by Year of Study",
    labels={
        "program": "Academic Program",
        "students": "Number of Students",
        "year": "Year of Study"
    }
)

fig.update_layout(
    width=950,
    height=600,
    showlegend=False
)

fig.update_xaxes(categoryorder="array", categoryarray=programs)

fig.show()

# Параметр animation_frame="year" создаёт отдельный кадр для каждого курса обучения. Пользователь может запускать анимацию и переключаться между кадрами с помощью ползунка.
# animation_frame="year" creates a separate frame for each year of study. Users can play the animation or switch between frames using the slider.

# **Сохранение интерактивного графика / Saving an Interactive Chart**

# Важное преимущество Plotly — возможность сохранять графики в HTML с сохранением интерактивности.
# An important advantage of Plotly is the ability to save charts as HTML files while preserving their interactivity.

In [ ]:
fig.write_html("Student_Visualization.html")

# TML-файл можно открыть в браузере без запуска Python. В нём сохраняются масштабирование, наведение курсора и другие интерактивные элементы.
# The HTML file can be opened in a browser without running Python. Zooming, hover information, and other interactive features are preserved.

# **Sunburst Chart** — иерархическая круговая диаграмма

# Sunburst позволяет отображать иерархические данные с несколькими уровнями вложенности. Например: образовательная программа → курс → получение стипендии.
# A Sunburst chart visualizes hierarchical data with multiple levels, such as academic program → year of study → scholarship status.

In [17]:
fig = px.sunburst(
    df_clean,
    path=["program", "year", "scholarship"],
    title="Student Distribution by Program, Year and Scholarship"
)

fig.show()

# **Treemap — древовидная карта**

# Treemap отображает иерархические категории в виде вложенных прямоугольников. Размер каждого прямоугольника отражает количество наблюдений.
# A Treemap displays hierarchical categories as nested rectangles, where rectangle size represents the number of observations.

In [18]:
fig = px.treemap(
    df_clean,
    path=["program", "year", "gender"],
    title="Student Distribution by Program, Year and Gender"
)

fig.show()

# **Violin Plot — скрипичная диаграмма**

# Violin plot объединяет характеристики boxplot с оценкой плотности распределения. Он позволяет увидеть, в каких диапазонах сосредоточено больше наблюдений.
# A violin plot combines boxplot statistics with an estimated probability density, showing where observations are concentrated.

In [ ]:
fig = px.violin(
    df_clean,
    x="program",
    y="final_score",
    color="program",
    box=True,
    points="all",
    title="Final Score Distribution by Program"
)

fig.show()

# **Parallel Coordinates — параллельные координаты**

# Один из наиболее интересных инструментов Plotly для многомерного анализа. Каждая линия представляет одного студента, а вертикальные оси — различные числовые признаки.
# Parallel Coordinates is a powerful visualization for multivariate analysis. Each line represents a student, while vertical axes represent numerical features.

In [19]:
fig = px.parallel_coordinates(
    df_clean,
    dimensions=[
        "math_score",
        "programming_score",
        "english_score",
        "attendance",
        "final_score"
    ],
    color="final_score",
    color_continuous_scale="Viridis",
    title="Multidimensional Student Performance Analysis"
)

fig.show()

# **Scatter Matrix — матрица диаграмм рассеивания**

# Scatter Matrix автоматически создаёт набор диаграмм рассеивания для всех пар выбранных признаков. Это удобно для предварительного исследования взаимосвязей.
# A Scatter Matrix automatically creates scatter plots for all pairs of selected features, making it useful for exploratory relationship analysis.

In [20]:
fig = px.scatter_matrix(
    df_clean,
    dimensions=[
        "math_score",
        "programming_score",
        "attendance",
        "final_score"
    ],
    color="program",
    title="Scatter Matrix of Student Performance"
)

fig.update_traces(diagonal_visible=False)
fig.show()

# **Interactive Dropdown — переключение данных на одном графике**

# Plotly позволяет создавать выпадающие списки, с помощью которых пользователь может менять отображаемый показатель без повторного запуска кода.
# Plotly supports dropdown menus that allow users to change the displayed metric without rerunning the code.

In [21]:
metrics = [
    "math_score",
    "programming_score",
    "english_score",
    "final_score"
]

fig = go.Figure()

for i, metric in enumerate(metrics):
    avg = df_clean.groupby("program")[metric].mean()

    fig.add_trace(
        go.Bar(
            x=avg.index,
            y=avg.values,
            name=metric,
            visible=(i == 0)
        )
    )

fig.update_layout(
    title="Average Student Scores by Program",
    xaxis_title="Program",
    yaxis_title="Average Score",
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label=metric,
                    method="update",
                    args=[{"visible": [
                        j == i for j in range(len(metrics))
                    ]}]
                )
                for i, metric in enumerate(metrics)
            ],
            direction="down"
        )
    ]
)

fig.show()